## Sel 1: Import Pustaka

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

import mlflow
import dagshub

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb

import optuna

import joblib
import contextlib
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import time
import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat Data (Resampling Harian)

In [2]:
# Pastikan path ini sesuai dengan posisi .env milikmu
load_dotenv('../.env', override=True)

# Ambil URL MLflow utuh 
db_url_mlflow = os.getenv("DATABASE_URL_DIRECT")

if not db_url_mlflow:
    raise ValueError("DATABASE_URL_DIRECT tidak ditemukan di .env!")

# Buat URL polos khusus untuk Pandas & SQLAlchemy
db_url_pandas = db_url_mlflow.split("?")[0]

print("Membuka jembatan ke Supabase...")
# Gunakan URL polos untuk engine
engine = create_engine(db_url_pandas)

# ==========================================
# LANGKAH 1: KUERI SQL JATAH RATA
# ==========================================
RENTANG_WAKTU = "1 year"

query_sql = f"""
    SELECT 
        waktu_aktual, 
        id_wilayah, 
        pm25, pm10, so2, co, no2, ozon
    FROM public.data_historis
    WHERE waktu_aktual >= NOW() - INTERVAL '{RENTANG_WAKTU}';
"""

ukuran_chunk = 50000
chunks = []

with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = 0"))
    conn.commit() 
    
    stream_conn = conn.execution_options(stream_results=True)
    
    for i, chunk in enumerate(pd.read_sql(text(query_sql), stream_conn, chunksize=ukuran_chunk)):
        print(f"-> Berhasil menyedot batch ke-{i+1} (hingga {(i+1) * ukuran_chunk} baris...)")
        chunks.append(chunk)

df_raw = pd.concat(chunks, ignore_index=True)
print(f"\n[+] Total data mentah ditarik: {df_raw.shape}")

Membuka jembatan ke Supabase...
-> Berhasil menyedot batch ke-1 (hingga 50000 baris...)
-> Berhasil menyedot batch ke-2 (hingga 100000 baris...)
-> Berhasil menyedot batch ke-3 (hingga 150000 baris...)
-> Berhasil menyedot batch ke-4 (hingga 200000 baris...)
-> Berhasil menyedot batch ke-5 (hingga 250000 baris...)
-> Berhasil menyedot batch ke-6 (hingga 300000 baris...)
-> Berhasil menyedot batch ke-7 (hingga 350000 baris...)

[+] Total data mentah ditarik: (327484, 8)


## Sel 3: Rekayasa Fitur

In [3]:
# =====================================================================
# 3. PREPROCESSING & REKAYASA FITUR (DATA ENGINEERING)
# =====================================================================
print("[~] Memulai proses pembersihan dan rekayasa fitur...")
df_clean = df_raw.copy()

kolom_waktu = 'waktu_aktual'
kolom_kota = 'id_wilayah'  
polutan = ['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon'] 

# ---------------------------------------------------------
# A. PRA-PEMROSESAN & RESAMPLE 
# ---------------------------------------------------------
df_clean[kolom_waktu] = pd.to_datetime(df_clean[kolom_waktu])
df_clean[kolom_waktu] = df_clean[kolom_waktu].dt.tz_localize(None)
df_clean.set_index(kolom_waktu, inplace=True)

df_hourly = df_clean.groupby(kolom_kota)[polutan].resample('H').mean().reset_index()

# ---------------------------------------------------------
# B. PEMBERSIHAN OUTLIER (Sensor Error) -> SEBELUM INTERPOLASI
# ---------------------------------------------------------
# Ubah nilai mustahil (<= 0) menjadi NaN agar ikut di-interpolasi
for col in polutan:
    df_hourly.loc[df_hourly[col] <= 0, col] = np.nan

# ---------------------------------------------------------
# C. INTERPOLASI & WINSORIZING
# ---------------------------------------------------------
print("Menambal data sensor yang bolong dengan interpolasi...")
df_hourly = df_hourly.groupby(kolom_kota, group_keys=False).apply(
    lambda group: group.interpolate(method='linear')
).reset_index(drop=True)

df_hourly = df_hourly.bfill().ffill()

# Potong lonjakan ekstrem (Winsorizing 99.9%)
for col in polutan:
    batas_atas = df_hourly[col].quantile(0.999)
    df_hourly[col] = np.where(df_hourly[col] > batas_atas, batas_atas, df_hourly[col])

# ---------------------------------------------------------
# D. PEMBUATAN FITUR TEMPORAL (LAG & ROLLING)
# ---------------------------------------------------------
print("Mengekstrak fitur temporal, Lag, dan Rolling Mean...")
df_hourly['jam'] = df_hourly[kolom_waktu].dt.hour
df_hourly['hari_dalam_minggu'] = df_hourly[kolom_waktu].dt.dayofweek
df_hourly['bulan'] = df_hourly[kolom_waktu].dt.month

fitur_baru = {}
for p in polutan:
    for lag in range(1, 25):
        fitur_baru[f'{p}_H-{lag}'] = df_hourly.groupby(kolom_kota)[p].shift(lag)
    
    fitur_baru[f'{p}_RollMean_72'] = df_hourly.groupby(kolom_kota)[p].transform(
        lambda x: x.rolling(window=72, min_periods=1).mean()
    )
    fitur_baru[f'{p}_RollMax_72'] = df_hourly.groupby(kolom_kota)[p].transform(
        lambda x: x.rolling(window=72, min_periods=1).max()
    )

df_features = pd.concat([df_hourly, pd.DataFrame(fitur_baru)], axis=1)

# ---------------------------------------------------------
# E. TARGET PREDIKSI (T+1 s/d T+24)
# ---------------------------------------------------------
print("Menciptakan target prediksi...")
target_cols = {}
for p in polutan:
    for t in range(1, 25):
        target_cols[f'target_{p}_{t}j'] = df_features.groupby(kolom_kota)[p].shift(-t)

df_model = pd.concat([df_features, pd.DataFrame(target_cols)], axis=1)

# ---------------------------------------------------------
# F. ONE-HOT ENCODING & FINALISASI
# ---------------------------------------------------------
df_model = pd.get_dummies(df_model, columns=[kolom_kota, 'jam', 'hari_dalam_minggu', 'bulan'], drop_first=True)

# Hapus baris NaN akibat fungsi Shift
df_model.dropna(inplace=True)
df_model.drop(columns=[kolom_waktu], inplace=True)

print(f"\n[+] Rekayasa fitur selesai! Bentuk data akhir: {df_model.shape}")

# =========================================================
# MENAMPILKAN DAFTAR KOLOM
# =========================================================
daftar_kolom = df_model.columns.tolist()
print("\n[+] Informasi Kolom:")
print(f"Total jumlah kolom: {len(daftar_kolom)}")
print("\n--- 20 Kolom Pertama (Biasanya Polutan Aktual & Lag) ---")
print(daftar_kolom[:20])

print("\n--- 20 Kolom Terakhir (Biasanya Hasil One-Hot Encoding) ---")
print(daftar_kolom[-20:])

# Jika butuh melihat semuanya (bisa sangat panjang), aktifkan baris di bawah ini:
# print("\n--- Daftar Keseluruhan Kolom ---")
# print(daftar_kolom)

# Menampilkan 5 baris teratas
display(df_model.head())

[~] Memulai proses pembersihan dan rekayasa fitur...
Menambal data sensor yang bolong dengan interpolasi...
Mengekstrak fitur temporal, Lag, dan Rolling Mean...
Menciptakan target prediksi...

[+] Rekayasa fitur selesai! Bentuk data akhir: (330714, 383)

[+] Informasi Kolom:
Total jumlah kolom: 383

--- 20 Kolom Pertama (Biasanya Polutan Aktual & Lag) ---
['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon', 'pm25_H-1', 'pm25_H-2', 'pm25_H-3', 'pm25_H-4', 'pm25_H-5', 'pm25_H-6', 'pm25_H-7', 'pm25_H-8', 'pm25_H-9', 'pm25_H-10', 'pm25_H-11', 'pm25_H-12', 'pm25_H-13', 'pm25_H-14']

--- 20 Kolom Terakhir (Biasanya Hasil One-Hot Encoding) ---
['jam_21', 'jam_22', 'jam_23', 'hari_dalam_minggu_1', 'hari_dalam_minggu_2', 'hari_dalam_minggu_3', 'hari_dalam_minggu_4', 'hari_dalam_minggu_5', 'hari_dalam_minggu_6', 'bulan_2', 'bulan_3', 'bulan_4', 'bulan_5', 'bulan_6', 'bulan_7', 'bulan_8', 'bulan_9', 'bulan_10', 'bulan_11', 'bulan_12']


,pm25,pm10,so2,co,no2,ozon,pm25_H-1,pm25_H-2,pm25_H-3,pm25_H-4,...,bulan_3,bulan_4,bulan_5,bulan_6,bulan_7,bulan_8,bulan_9,bulan_10,bulan_11,bulan_12
24,31.84,38.88,3.83,367.29,5.16,52.43,24.96,21.27,20.30,19.34,...,False,False,False,True,False,False,False,False,False,False
25,34.40,40.20,3.95,347.31,4.65,53.71,31.84,24.96,21.27,20.30,...,False,False,False,True,False,False,False,False,False,False
26,35.03,40.37,3.81,354.69,4.81,52.45,34.40,31.84,24.96,21.27,...,False,False,False,True,False,False,False,False,False,False
27,34.50,40.03,3.75,374.80,5.37,51.65,35.03,34.40,31.84,24.96,...,False,False,False,True,False,False,False,False,False,False
28,35.14,42.58,3.93,480.40,7.51,52.15,34.50,35.03,34.40,31.84,...,False,False,False,True,False,False,False,False,False,False


In [7]:
# =====================================================================
# CEK VISUAL: BUKTI REKAYASA FITUR (TEMPORAL, SHIFT & ROLLING)
# =====================================================================
kota_contoh = 21       
polutan_contoh = 'pm25' 

# Menambahkan 'bulan' dan 'hari_dalam_minggu' ke dalam susunan kolom
kolom_pilihan = [
    'waktu_aktual',
    'bulan',                 # Fitur Pola Musiman Tahunan
    'hari_dalam_minggu',     # Fitur Pola Mingguan (0=Senin, 6=Minggu)
    'jam',                   # Fitur Pola Harian
    polutan_contoh,          # Nilai Aktual (Target)
    f'{polutan_contoh}_H-1', # Memori 1 Jam lalu (Shift 1)
    f'{polutan_contoh}_H-2', # Memori 2 Jam lalu (Shift 2)
    f'{polutan_contoh}_H-24',# Memori Kemarin (Shift 24)
    f'{polutan_contoh}_RollMean_72', # Konteks Makro 3 Hari
    f'{polutan_contoh}_RollMax_72'
]

# Ambil cuplikan berurutan untuk 1 kota
df_showcase = df_features[df_features['id_wilayah'] == kota_contoh][kolom_pilihan].tail(5).copy()

# Membulatkan nilai agar tabel rapi saat di-screenshot
df_showcase = df_showcase.round(1)

print(f"--- Visualisasi Fitur AI untuk Kota ID {kota_contoh} (Polutan: {polutan_contoh.upper()}) ---")
display(df_showcase)

--- Visualisasi Fitur AI untuk Kota ID 21 (Polutan: PM25) ---


,waktu_aktual,bulan,hari_dalam_minggu,jam,pm25,pm25_H-1,pm25_H-2,pm25_H-24,pm25_RollMean_72,pm25_RollMax_72
183766,2026-06-02 14:00:00,6,1,14,2.8,2.5,2.3,1.5,2.0,2.9
183767,2026-06-02 15:00:00,6,1,15,2.8,2.8,2.5,1.4,2.0,2.9
183768,2026-06-02 16:00:00,6,1,16,2.8,2.8,2.8,1.3,2.0,2.9
183769,2026-06-02 17:00:00,6,1,17,2.8,2.8,2.8,1.2,2.0,2.9
183770,2026-06-02 18:00:00,6,1,18,2.8,2.8,2.8,1.2,2.0,2.9


## Sel 4: Pemisahan Data, Penentuan Proxy Task & Normalisasi (Scaling)

In [10]:
print("[~] Memisahkan Fitur (X) dan Target (y)...")

# 1. Ambil X dan y
kolom_target = [col for col in df_model.columns if col.startswith('target_')]
kolom_fitur = [col for col in df_model.columns if not col.startswith('target_')]

X = df_model[kolom_fitur]
y = df_model[kolom_target]

# 2. Splitting Data (80% Train, 20% Test, Time Series)
print("[~] Membelah data... ")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. PENENTUAN PROXY TASK: Kita hanya akan mengadu performa model untuk memprediksi PM2.5 1 Jam ke Depan (T+1)
print("[~] Mengekstrak Proxy Task (Target: PM2.5 T+1)...")
target_col = 'target_pm25_1j'
y_train_proxy = y_train[target_col].values
y_test_proxy = y_test[target_col].values

# 4. SCALING (Wajib untuk algoritma Jarak seperti SVM agar adil melawan Tree)
print("[~] Menjalankan StandardScaler pada fitur...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Menerapkan Transformasi Logaritmik (Log1p) pada target
y_train_proxy_log = np.log1p(y_train_proxy)
y_test_proxy_log = np.log1p(y_test_proxy)

print("[+] Data siap untuk dimasukkan ke arena pertandingan!")


[~] Memisahkan Fitur (X) dan Target (y)...
[~] Membelah data... 
[~] Mengekstrak Proxy Task (Target: PM2.5 T+1)...
[~] Menjalankan StandardScaler pada fitur...
[+] Data siap untuk dimasukkan ke arena pertandingan!


## Sel 5: Koneksi MLFlow DagsHub

In [11]:
# Ambil Konfigurasi DagsHub dari .env
repo_owner = os.getenv("DAGSHUB_REPO_OWNER")
repo_name = os.getenv("DAGSHUB_REPO_NAME")

if not repo_owner or not repo_name:
    raise ValueError("Identitas DAGSHUB_REPO_OWNER atau DAGSHUB_REPO_NAME belum di-set di file .env!")

print(f"Connecting to DagsHub MLflow Server ({repo_owner}/{repo_name})...")
dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)

# Set Eksperimen Khusus AutoML Baseline
mlflow.set_experiment("Baseline_AutoML_ProxyTask")


Connecting to DagsHub MLflow Server (tegarkusuma12/Web-ISPU)...


Initialized MLflow to track repo "tegarkusuma12/Web-ISPU"

Repository tegarkusuma12/Web-ISPU initialized!

<Experiment: artifact_location='mlflow-artifacts:/bf3a5a95b8fe48a2adcc4e0cdfb36d82', creation_time=1780422427981, experiment_id='8', last_update_time=1780422427981, lifecycle_stage='active', name='Baseline_AutoML_ProxyTask', tags={}, trace_location=None, workspace='default'>

## Sel 6: Eksperimen Komparasi Model

In [13]:
print("=== MEMULAI PERTANDINGAN PROXY TASK (TARGET: PM2.5 T+1) ===\n")

# Karena SVM O(N^2), dataset 330k baris akan freeze. Kita sampling ambil 5000 baris data akhir untuk training yang efisien
# (Ini sangat adil karena XGB/RF juga akan dilatih dengan 20% data yang sama agar apple-to-apple)
sample_size = 20000
print(f"Melakukan sampling {sample_size} baris data akhir agar SVM dapat menyelesaikan komputasi dalam waktu wajar...")
X_train_sample = X_train_scaled[-sample_size:]
y_train_sample = y_train_proxy_log[-sample_size:]

# Definisi Kandidat Model dengan parameter seimbang
model_kandidat = {
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=6, random_state=42, tree_method='hist', n_jobs=-1),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1),
    'SVM': SVR(kernel='rbf', C=1.0) # SVR default menggunakan kernel RBF
}

hasil_eksperimen = []

for nama_model, model in model_kandidat.items():
    print(f"\n---> Sedang melatih {nama_model}...")
    start_time = time.time()
    
    with mlflow.start_run(run_name=f"{nama_model}_Baseline_PM25_T1"):
        # Pelatihan model
        model.fit(X_train_sample, y_train_sample)
        waktu_latih = time.time() - start_time
        
        # Prediksi
        y_pred_log = model.predict(X_test_scaled)
        
        # Kembalikan wujud asli (Microgram)
        y_pred_asli = np.expm1(y_pred_log)
        y_test_asli = np.expm1(y_test_proxy_log)
        
        # Evaluasi
        mae = mean_absolute_error(y_test_asli, y_pred_asli)
        rmse = np.sqrt(mean_squared_error(y_test_asli, y_pred_asli))
        r2 = r2_score(y_test_asli, y_pred_asli)
        
        # Mencatat ke MLFlow
        mlflow.log_param("model_type", nama_model)
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)
        mlflow.log_metric("training_time_seconds", waktu_latih)
        
        print(f"[+] {nama_model} selesai! (Waktu: {waktu_latih:.2f} detik)")
        print(f"    MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}")
        
        hasil_eksperimen.append({
            'Model': nama_model,
            'Waktu Training (s)': round(waktu_latih, 2),
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2),
            'R-Squared': round(r2, 4)
        })

print("\n=== KLASEMEN AKHIR EKSPERIMEN ===")
df_hasil = pd.DataFrame(hasil_eksperimen).sort_values(by='MAE', ascending=True)
display(df_hasil)


=== MEMULAI PERTANDINGAN PROXY TASK (TARGET: PM2.5 T+1) ===

Melakukan sampling 5000 baris data akhir agar SVM dapat menyelesaikan komputasi dalam waktu wajar...

---> Sedang melatih XGBoost...
[+] XGBoost selesai! (Waktu: 2.28 detik)
    MAE: 0.25 | RMSE: 0.89 | R2: 0.9819
🏃 View run XGBoost_Baseline_PM25_T1 at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/8/runs/a205487a2f754625b2d0590c590bbaa7
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/8

---> Sedang melatih RandomForest...
[+] RandomForest selesai! (Waktu: 18.01 detik)
    MAE: 0.31 | RMSE: 0.95 | R2: 0.9791
🏃 View run RandomForest_Baseline_PM25_T1 at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/8/runs/1a47f09356734e29b39b8392ae217d28
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/8

---> Sedang melatih SVM...
[+] SVM selesai! (Waktu: 40.21 detik)
    MAE: 0.75 | RMSE: 2.37 | R2: 0.8703
🏃 View run SVM_Baseline_PM

,Model,Waktu Training (s),MAE,RMSE,R-Squared
0,XGBoost,2.28,0.25,0.89,0.9819
1,RandomForest,18.01,0.31,0.95,0.9791
2,SVM,40.21,0.75,2.37,0.8703
